In [ ]:
# ============================================================
#  环境配置
#  - Colab 用户：取消注释下方 Colab 区块
#  - 本地 Jupyter 用户：直接运行 Local 区块
# ============================================================

# ── Colab 环境（取消注释后运行） ──
# !pip install torch matplotlib numpy -q

# ── 本地 Jupyter 环境 ──
import subprocess, sys

def _install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

for pkg in ["torch", "matplotlib", "numpy"]:
    _install(pkg)

In [ ]:
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Non-Local 代码实战：学习实现 vs 工程实现

如果只依赖卷积，远距离时空关系往往要靠很多层间接传播；
Non-Local 的核心想法更直接：**让任意两个时空位置在一层内就能交换信息**。

本 Notebook 以论文 *Non-Local Neural Networks* (Wang et al., CVPR 2018) 为主线，
用一个可以快速运行的小型视频分类任务，把“论文公式”翻译成“可运行代码”。

建议阅读顺序：
1. 先抓主线：Non-Local 到底在解决什么问题；
2. 再看学习路径：`theta / phi / g`、关系矩阵、残差连接如何落地；
3. 最后看工程路径：同一思想怎样变成更短、更稳的工程实现。

| | 学习路径 (Learning) | 工程路径 (Engineering) |
|---|---|---|
| 目标 | 看懂 Non-Local 的数学结构、shape 流转与插入方式 | 掌握在真实工程里更常见的高层 API 写法 |
| 实现方式 | L1：手写 `theta/phi/g` 投影、关系矩阵与残差连接，并训练一个小模型 | E2/E3：用 `nn.MultiheadAttention(batch_first=True)` 复现同类全局聚合思想 |
| 代码量 | ~180 行 | ~90 行 |
| 适合场景 | 面试准备、论文精读、调试 shape | 快速原型、模块替换、工程迁移 |

> 这两条路径不是在竞争“谁更高级”，而是在解决两个不同问题：
> 前者帮你**真正看懂**，后者帮你**更快写出来并稳定运行**。

## i. 论文与任务背景

### 这篇论文在回答什么问题？

传统卷积网络非常擅长提取局部模式，但它天然偏向“看邻域”。
如果两个关键区域在空间上相距很远，或者同一目标跨越多帧才显现关联，
网络通常需要依赖很多层卷积、很大的感受野，才能把这些信息慢慢传过去。

Non-Local 的切入点很直接：
> 不再要求信息必须沿局部邻域逐层传播，而是让任意位置都能**直接**参考全局所有位置。

### 为什么用视频分类来讲 Non-Local？

视频正是 Non-Local 最自然的应用场景之一。
对于输入 `(B, C, T, H, W)`：
- 3D 卷积擅长建模局部的时空邻域；
- Non-Local 擅长把远距离帧、远距离空间位置直接关联起来；
- 两者组合起来，正好能体现“局部建模 + 全局建模”的分工。

因此，本 Notebook 不是为了复现论文的大规模实验，
而是为了把这个模块放到一个足够小、但又足够有代表性的任务里，让你真正看清它在做什么。

### 读完这一节，你应该能回答三个问题

1. Non-Local 到底比普通卷积“多做了什么”？
2. `theta / phi / g` 各自扮演什么角色？
3. 如果在工程里不想手写所有矩阵乘法，怎样用高层 API 复现同样的思想？

### 本 Notebook 的范围

本 Notebook 聚焦 **Embedded Gaussian 版本** 的 Non-Local Block：
- 覆盖：核心公式、手写实现、简洁工程实现、训练结果对比、面试表达；
- 不覆盖：论文所有变体的大规模实验复现、Kinetics/COCO 真数据训练、下采样版 `phi/g` 的完整工程优化。

## ii. 最小必要理论

这一节只保留“写代码时真正会用到”的理论，不重复 companion `.md` 里的完整论文解读。
目标很明确：**让你能把公式和代码一一对应起来。**

### 1) Non-Local 响应的基本形式

对于输入特征 `x in R^(B, C, T, H, W)`，令 `N = T * H * W`。
论文把第 `i` 个位置的响应写成：

$$
y_i = \frac{1}{\mathcal{C}(x)} \sum_j f(x_i, x_j) g(x_j), \quad
z_i = W_z y_i + x_i
$$

这个公式可以分成两步理解：
- 先算“位置 `i` 应该从每个位置 `j` 吸收多少信息”；
- 再把所有位置的值 `g(x_j)` 按权重加权求和，最后通过残差回写到原特征上。

### 2) Embedded Gaussian：最常用、也最容易实现的版本

在论文最常用的实现里：

$$
f(x_i, x_j) = \exp(\theta(x_i)^\top \phi(x_j))
$$

于是矩阵形式可以写成：

$$
\mathbf{z} = W_z \cdot \text{softmax}(\theta(\mathbf{x})^\top \phi(\mathbf{x})) \cdot g(\mathbf{x}) + \mathbf{x}
$$

如果你熟悉 Transformer，可以把它先粗略理解为：
- `theta(x)`：当前位置“想查什么”；
- `phi(x)`：所有位置“能提供什么索引”；
- `g(x)`：所有位置“真正被聚合的值”。

### 3) 为什么它比卷积更“全局”？

卷积默认只看局部邻域，远距离信息必须靠多层传播才能到达。
Non-Local 不一样：**一次关系计算就能把任意两个位置连接起来**。
这就是它最核心的价值，也是它计算更贵的根源。

### 4) 写代码时最容易弄错的部分：shape

- 输入：`(B, C, T, H, W)`
- 展平后位置数：`N = T * H * W`
- `theta/phi/g`：`(B, C_mid, N)`
- 注意力矩阵：`(B, N, N)`
- 聚合输出：`(B, C_mid, N)` → reshape 回 `(B, C_mid, T, H, W)` → 投影回 `(B, C, T, H, W)`

后面你会发现，真正难的不是公式本身，而是：
> **每一次 transpose / reshape 到底有没有保持语义正确。**

## 组件映射表

| 论文组件 | 学习路径覆盖 | 工程库 / API 对应 | 状态 |
|---|---|---|---|
| `theta(x)` 线性嵌入 | `Conv3d(1x1x1)` 手写实现 | `nn.MultiheadAttention` 内部 `W_Q` | Runnable |
| `phi(x)` 线性嵌入 | `Conv3d(1x1x1)` 手写实现 | `nn.MultiheadAttention` 内部 `W_K` | Runnable |
| `g(x)` 值映射 | `Conv3d(1x1x1)` 手写实现 | `nn.MultiheadAttention` 内部 `W_V` | Runnable |
| `softmax(theta^T phi)` 关系计算 | `torch.bmm + softmax` 显式写出 | `nn.MultiheadAttention` 内部缩放点积注意力 | Runnable |
| `W_z` 输出投影 | `Conv3d(1x1x1)` 手写实现 | `nn.MultiheadAttention.out_proj` + reshape | Runnable |
| 残差连接 `+ x` | 显式实现 | 显式实现 | Runnable |
| 长距离时空依赖建模 | 插入 3D CNN 后训练验证 | 插入 3D CNN 后训练验证 | Runnable |
| 多头机制 | 不使用，保留论文原始单关系矩阵思路 | 使用高层 API 的多头版本近似表达 | Illustrative |
| 下采样优化 (`phi/g` pooling) | 本 notebook 仅文字说明 | 未实现 | Explain-only |

## 1. 数据准备

为了让这份 Notebook 能在 Colab 免费环境和普通 CPU 上直接运行，
这里不使用真实视频数据集，而是构造一个**可控的合成视频分类任务**。

这样做有三个好处：
- **运行门槛低**：不需要下载大数据集，也不会把注意力分散到数据清洗上；
- **教学信号清晰**：每一类样本都对应一种明确的时空模式；
- **更容易观察模块作用**：我们更容易把性能差异归因到“是否显式建模了全局关系”。

3 个类别分别对应 3 种模式：
- 类别 0：水平条纹随时间向下移动；
- 类别 1：垂直条纹随时间向右移动；
- 类别 2：中心方块逐步扩张。

你可以把它理解成一个“最小可运行的时空关系识别任务”：
虽然它远没有真实视频复杂，但已经足够承载 Non-Local 的核心思想。

In [ ]:
class SyntheticVideoDataset(Dataset):
    """合成视频分类数据集。"""

    def __init__(self, n_samples, num_classes, n_frames, img_h, img_w, noise_std=0.05):
        self.videos = []
        self.labels = []

        for i in range(n_samples):
            label = i % num_classes
            video = torch.zeros(1, n_frames, img_h, img_w)

            for t in range(n_frames):
                if label == 0:
                    row = (2 + t) % img_h
                    video[0, t, row:min(row + 3, img_h), :] = 1.0
                elif label == 1:
                    col = (2 + t) % img_w
                    video[0, t, :, col:min(col + 3, img_w)] = 1.0
                else:
                    size = 2 + t // 2
                    cy, cx = img_h // 2, img_w // 2
                    y1, y2 = max(0, cy - size), min(img_h, cy + size)
                    x1, x2 = max(0, cx - size), min(img_w, cx + size)
                    video[0, t, y1:y2, x1:x2] = 1.0

            video = video + noise_std * torch.randn_like(video)
            self.videos.append(video)
            self.labels.append(label)

        self.videos = torch.stack(self.videos)          # (N, 1, T, H, W)
        self.labels = torch.tensor(self.labels).long()  # (N,)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.videos[idx], self.labels[idx]

In [ ]:
# ── 数据超参数 ──
NUM_CLASSES = 3
N_FRAMES = 8
IMG_H, IMG_W = 16, 16
TRAIN_SAMPLES = 600
TEST_SAMPLES = 120
BATCH_SIZE = 32

train_ds = SyntheticVideoDataset(TRAIN_SAMPLES, NUM_CLASSES, N_FRAMES, IMG_H, IMG_W)
test_ds = SyntheticVideoDataset(TEST_SAMPLES, NUM_CLASSES, N_FRAMES, IMG_H, IMG_W)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train samples: {len(train_ds)}")
print(f"Test samples : {len(test_ds)}")

In [ ]:
videos, labels = next(iter(train_loader))
print(f"Video batch shape: {videos.shape}")   # (B, 1, T, H, W)
print(f"Label batch shape: {labels.shape}")   # (B,)
print(f"Label examples   : {labels[:8].tolist()}")

fig, axes = plt.subplots(1, 4, figsize=(10, 3))
for i in range(4):
    axes[i].imshow(videos[0, 0, i].numpy(), cmap="gray")
    axes[i].set_title(f"Frame {i}")
    axes[i].axis("off")
plt.tight_layout()
plt.show()

## 2. 共享组件

在真正比较两条路径之前，先把训练协议统一。
这样后面的差异就更容易归因到“模块写法不同”，而不是“训练设置不同”。

这里统一的内容包括：
- 超参数；
- 训练循环；
- 评估函数；
- 曲线可视化方式。

这一步看似普通，但很重要。
如果没有统一协议，后面的对比就会混入太多无关变量，教程也会失去说服力。

In [ ]:
# ── 超参数（两条路径共用，集中管理） ──
BASE_CHANNELS = 16
HIDDEN_CHANNELS = 32
FINAL_CHANNELS = 64
DROPOUT = 0.1
LR = 1e-3
NUM_EPOCHS = 8

print(f"Device       : {device}")
print(f"Batch size   : {BATCH_SIZE}")
print(f"Epochs       : {NUM_EPOCHS}")
print(f"Frames       : {N_FRAMES}")
print(f"Image size   : {IMG_H}x{IMG_W}")

In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def train_model(model, train_loader, epochs=NUM_EPOCHS, lr=LR):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    history = []

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        running_correct = 0
        running_total = 0

        for videos, labels in train_loader:
            videos = videos.to(device)
            labels = labels.to(device)

            logits = model(videos)                    # (B, num_classes)
            loss = criterion(logits, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            running_correct += (logits.argmax(dim=1) == labels).sum().item()
            running_total += labels.size(0)

        avg_loss = running_loss / len(train_loader)
        avg_acc = running_correct / running_total
        history.append((avg_loss, avg_acc))

        print(f"Epoch {epoch + 1:02d}/{epochs} | loss={avg_loss:.4f} | acc={avg_acc:.2%}")

    return history


@torch.no_grad()
def evaluate_model(model, data_loader):
    model = model.to(device)
    model.eval()
    correct = 0
    total = 0

    for videos, labels in data_loader:
        videos = videos.to(device)
        labels = labels.to(device)
        logits = model(videos)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)

    return correct / total

In [ ]:
def plot_histories(history_dict):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    for name, history in history_dict.items():
        losses = [x[0] for x in history]
        accs = [x[1] for x in history]
        axes[0].plot(losses, label=name)
        axes[1].plot(accs, label=name)

    axes[0].set_title("Training Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()

    axes[1].set_title("Training Accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy")
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()

    plt.tight_layout()
    plt.show()

## iii. 学习路径：从零实现 Non-Local Block

这一部分采用 **L1：可完整训练的小模型**。
目标不是把 backbone 写得更复杂，而是把论文里最关键的计算链条完整落地。

阅读这一节时，建议盯住三件事：
1. **关系矩阵是谁算出来的**；
2. **shape 在哪里发生变化**；
3. **为什么 residual 能让模块具备“即插即用”的性质**。

实现顺序按数据流展开：

```text
input video
  -> conv stem
  -> Non-Local block
  -> deeper convs
  -> global pooling
  -> classifier
```

你会看到，真正重要的不是代码行数，而是：
> 我们有没有把“全局关系建模”这件事用最直白的方式写出来。

### 学习组件 1：Non-Local Block 的投影与关系矩阵

先不要急着把它看成“复杂注意力模块”，可以先把它拆成三个更朴素的问题：

1. **谁在发起查询？** —— `theta(x)`
2. **谁在提供可被匹配的索引？** —— `phi(x)`
3. **真正被聚合回来的内容是什么？** —— `g(x)`

输入 `x` 的 shape 是 `(B, C, T, H, W)`。
令 `N = T * H * W`，并设中间通道数 `C_mid = C / 2`。

先通过三个 `1x1x1` 卷积得到：

$$
\theta(x), \phi(x), g(x) \in \mathbb{R}^{B \times C_{mid} \times T \times H \times W}
$$

再把时空维度展平：

$$
\theta(x), \phi(x), g(x) \rightarrow \mathbb{R}^{B \times C_{mid} \times N}
$$

这时关系矩阵就可以写成：

$$
A = \text{softmax}(\theta(x)^\top \phi(x)) \in \mathbb{R}^{B \times N \times N}
$$

这里的 `A[i, j]` 可以直观理解为：
> **位置 `i` 觉得位置 `j` 对自己有多重要。**

随后再用这个关系矩阵去加权聚合 `g(x)`，并通过 `W_z` 投影回原通道，
最后与输入做残差相加。

所以从实现角度看，Non-Local 并不神秘：
它本质上就是“先算全局关系，再按关系加权聚合，再回写原特征”。

In [ ]:
class NonLocalBlock3D(nn.Module):
    """Embedded Gaussian version of Non-Local Block for 3D features."""

    def __init__(self, in_channels):
        super().__init__()
        inter_channels = max(1, in_channels // 2)
        self.inter_channels = inter_channels

        self.theta = nn.Conv3d(in_channels, inter_channels, kernel_size=1)  # (B,C,T,H,W) -> (B,Cmid,T,H,W)
        self.phi = nn.Conv3d(in_channels, inter_channels, kernel_size=1)    # (B,C,T,H,W) -> (B,Cmid,T,H,W)
        self.g = nn.Conv3d(in_channels, inter_channels, kernel_size=1)      # (B,C,T,H,W) -> (B,Cmid,T,H,W)
        self.W_z = nn.Conv3d(inter_channels, in_channels, kernel_size=1)    # (B,Cmid,T,H,W) -> (B,C,T,H,W)

        nn.init.zeros_(self.W_z.weight)
        nn.init.zeros_(self.W_z.bias)

    def forward(self, x):
        # x: (B, C, T, H, W)
        b, c, t, h, w = x.shape
        n = t * h * w

        theta = self.theta(x).view(b, self.inter_channels, n)               # (B, Cmid, N)
        phi = self.phi(x).view(b, self.inter_channels, n)                   # (B, Cmid, N)
        g = self.g(x).view(b, self.inter_channels, n)                       # (B, Cmid, N)

        affinity = torch.bmm(theta.transpose(1, 2), phi)                    # (B, N, Cmid) @ (B, Cmid, N) -> (B, N, N)
        attention = F.softmax(affinity, dim=-1)                             # (B, N, N)

        y = torch.bmm(g, attention.transpose(1, 2))                         # (B, Cmid, N)
        y = y.view(b, self.inter_channels, t, h, w)                         # (B, Cmid, T, H, W)
        z = self.W_z(y) + x                                                 # (B, C, T, H, W)
        return z

In [ ]:
nl_block = NonLocalBlock3D(in_channels=BASE_CHANNELS)
dummy_x = torch.randn(2, BASE_CHANNELS, 4, 8, 8)
dummy_y = nl_block(dummy_x)

print(f"Input shape : {dummy_x.shape}")
print(f"Output shape: {dummy_y.shape}")
print(f"Max |y-x| at init: {(dummy_y - dummy_x).abs().max().item():.8f}")

### 学习组件 2：将 Non-Local 插入 3D CNN

接下来不直接上完整大模型，而是先建立两个最小可比较对象：
- 一个 baseline 3D CNN；
- 一个在中间插入 Non-Local 的 3D CNN。

关键问题不是“能不能插进去”，而是：
> **应该插在什么位置，既能体现全局关系，又不会把计算量拉爆？**

这里选择在第一层池化之后插入，原因非常典型：

- 原始输入尺度下，`N = 8 * 16 * 16 = 2048`；
- 如果直接做全局关系，单个样本的关系矩阵就是 `2048 x 2048`，开销明显偏大；
- 第一层池化后，特征图变成 `(T=4, H=8, W=8)`，此时 `N = 256`；
- `256 x 256` 的关系矩阵已经足够小，CPU 也能稳定跑。

这正是论文和工程实践里都很常见的思路：
**先用局部卷积做初步特征抽取，再在较低分辨率上引入全局关系。**

In [ ]:
class Baseline3DCNN(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.conv1 = nn.Conv3d(1, BASE_CHANNELS, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm3d(BASE_CHANNELS)
        self.pool1 = nn.MaxPool3d(2)

        self.conv2 = nn.Conv3d(BASE_CHANNELS, HIDDEN_CHANNELS, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm3d(HIDDEN_CHANNELS)
        self.pool2 = nn.MaxPool3d(2)

        self.conv3 = nn.Conv3d(HIDDEN_CHANNELS, FINAL_CHANNELS, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm3d(FINAL_CHANNELS)
        self.gap = nn.AdaptiveAvgPool3d(1)
        self.fc = nn.Linear(FINAL_CHANNELS, num_classes)

    def forward(self, x):
        # x: (B, 1, 8, 16, 16)
        x = self.pool1(F.relu(self.bn1(self.conv1(x))))   # (B, 16, 4, 8, 8)
        x = self.pool2(F.relu(self.bn2(self.conv2(x))))   # (B, 32, 2, 4, 4)
        x = F.relu(self.bn3(self.conv3(x)))               # (B, 64, 2, 4, 4)
        x = self.gap(x)                                   # (B, 64, 1, 1, 1)
        x = x.flatten(1)                                  # (B, 64)
        return self.fc(x)                                 # (B, 3)


class NonLocal3DCNN(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.conv1 = nn.Conv3d(1, BASE_CHANNELS, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm3d(BASE_CHANNELS)
        self.pool1 = nn.MaxPool3d(2)
        self.non_local = NonLocalBlock3D(BASE_CHANNELS)

        self.conv2 = nn.Conv3d(BASE_CHANNELS, HIDDEN_CHANNELS, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm3d(HIDDEN_CHANNELS)
        self.pool2 = nn.MaxPool3d(2)

        self.conv3 = nn.Conv3d(HIDDEN_CHANNELS, FINAL_CHANNELS, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm3d(FINAL_CHANNELS)
        self.gap = nn.AdaptiveAvgPool3d(1)
        self.fc = nn.Linear(FINAL_CHANNELS, num_classes)

    def forward(self, x):
        # x: (B, 1, 8, 16, 16)
        x = self.pool1(F.relu(self.bn1(self.conv1(x))))   # (B, 16, 4, 8, 8)
        x = self.non_local(x)                             # (B, 16, 4, 8, 8)
        x = self.pool2(F.relu(self.bn2(self.conv2(x))))   # (B, 32, 2, 4, 4)
        x = F.relu(self.bn3(self.conv3(x)))               # (B, 64, 2, 4, 4)
        x = self.gap(x)                                   # (B, 64, 1, 1, 1)
        x = x.flatten(1)                                  # (B, 64)
        return self.fc(x)                                 # (B, 3)

In [ ]:
baseline_model = Baseline3DCNN()
non_local_model = NonLocal3DCNN()

dummy_video = torch.randn(2, 1, N_FRAMES, IMG_H, IMG_W)
print(f"Baseline output shape   : {baseline_model(dummy_video).shape}")
print(f"Non-Local output shape : {non_local_model(dummy_video).shape}")
print(f"Baseline params        : {count_parameters(baseline_model):,}")
print(f"Non-Local params       : {count_parameters(non_local_model):,}")

### 训练学习路径模型

下面分别训练：
- **Baseline 3D CNN**：只有局部卷积；
- **3D CNN + Non-Local**：在中间层显式建立全局时空关系。

这一节建议重点观察两个现象：
1. loss 下降速度是否发生变化；
2. test accuracy 是否因为引入全局关系而得到提升。

需要注意的是：
这只是一个小型教学任务，单次运行的结果不能当作论文结论。
但它足以回答一个更重要的问题：
> 在完全相同的数据与训练协议下，显式全局关系有没有帮助模型更快抓住跨帧模式？

In [ ]:
print("=== Train baseline model ===")
learning_baseline = Baseline3DCNN()
history_baseline = train_model(learning_baseline, train_loader)
acc_baseline = evaluate_model(learning_baseline, test_loader)
print(f"Baseline test accuracy: {acc_baseline:.2%}")

print("\n=== Train Non-Local model ===")
learning_non_local = NonLocal3DCNN()
history_non_local = train_model(learning_non_local, train_loader)
acc_non_local = evaluate_model(learning_non_local, test_loader)
print(f"Non-Local test accuracy: {acc_non_local:.2%}")

## iv. 工程路径：用高层 API 复现全局关系

如果学习路径解决的是“我到底有没有真正看懂”，
那么工程路径解决的就是另一个现实问题：
> **当我已经理解思想后，怎样更快、更稳地把它写进工程里？**

这一部分不再显式手写 `theta/phi/g` 和 `torch.bmm`，而是直接使用 PyTorch 的 `nn.MultiheadAttention`。

根据 PyTorch 官方文档，`batch_first=True` 时输入输出形状为 `(batch, seq, feature)`。
所以这里要先把 `(B, C, T, H, W)` 变成 `(B, N, C_mid)`，把每个时空位置当成一个 token。

为了让这条路径尽量和学习路径可比，这里特意做了三件事：
- 先用 `1x1x1 Conv3d` 把通道压到 `C_mid`；
- 使用 **single-head** attention，而不是多头；
- 用零初始化的输出投影映回原通道，保留“初始近似恒等映射”的性质。

### 对应关系

| 学习路径手写组件 | 工程路径高层 API | 说明 |
|---|---|---|
| 通道降维 `C -> C_mid` | `pre` 1x1x1 Conv3d | 对齐论文里的中间通道压缩 |
| `theta/phi/g` 关系建模 | `nn.MultiheadAttention` 内部 Q/K/V 投影 | 线性映射被封装 |
| `softmax(theta^T phi)` | scaled dot-product attention | 关系矩阵计算被封装 |
| `W_z` | `post` 1x1x1 Conv3d | 输出投影回原通道，并做零初始化 |
| `+ x` residual | 显式保留 | 残差连接仍由我们控制 |

这条路径不是论文的逐行等价实现，
但它展示了一个非常重要的工程能力：
**把论文思想迁移到成熟 API 上，而不是永远停留在手写版本。**

In [ ]:
class NonLocalBlockMHA(nn.Module):
    """A closer engineering approximation using MHA + channel reduction."""

    def __init__(self, in_channels):
        super().__init__()
        inter_channels = max(1, in_channels // 2)
        self.inter_channels = inter_channels

        self.pre = nn.Conv3d(in_channels, inter_channels, kernel_size=1)   # (B,C,T,H,W) -> (B,Cmid,T,H,W)
        self.attn = nn.MultiheadAttention(
            embed_dim=inter_channels,
            num_heads=1,
            dropout=0.0,
            batch_first=True,
        )
        self.post = nn.Conv3d(inter_channels, in_channels, kernel_size=1)  # (B,Cmid,T,H,W) -> (B,C,T,H,W)

        nn.init.zeros_(self.post.weight)
        nn.init.zeros_(self.post.bias)

    def forward(self, x):
        # x: (B, C, T, H, W)
        b, c, t, h, w = x.shape
        n = t * h * w

        reduced = self.pre(x)                                              # (B, Cmid, T, H, W)
        tokens = reduced.reshape(b, self.inter_channels, n).transpose(1, 2)  # (B, N, Cmid)
        out, _ = self.attn(tokens, tokens, tokens)                         # (B, N, Cmid)
        out = out.transpose(1, 2).contiguous().reshape(b, self.inter_channels, t, h, w)
        out = self.post(out)                                               # (B, C, T, H, W)
        return out + x


class NonLocal3DCNN_MHA(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.conv1 = nn.Conv3d(1, BASE_CHANNELS, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm3d(BASE_CHANNELS)
        self.pool1 = nn.MaxPool3d(2)
        self.non_local = NonLocalBlockMHA(BASE_CHANNELS)

        self.conv2 = nn.Conv3d(BASE_CHANNELS, HIDDEN_CHANNELS, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm3d(HIDDEN_CHANNELS)
        self.pool2 = nn.MaxPool3d(2)

        self.conv3 = nn.Conv3d(HIDDEN_CHANNELS, FINAL_CHANNELS, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm3d(FINAL_CHANNELS)
        self.gap = nn.AdaptiveAvgPool3d(1)
        self.fc = nn.Linear(FINAL_CHANNELS, num_classes)

    def forward(self, x):
        x = self.pool1(F.relu(self.bn1(self.conv1(x))))            # (B, 16, 4, 8, 8)
        x = self.non_local(x)                                      # (B, 16, 4, 8, 8)
        x = self.pool2(F.relu(self.bn2(self.conv2(x))))            # (B, 32, 2, 4, 4)
        x = F.relu(self.bn3(self.conv3(x)))                        # (B, 64, 2, 4, 4)
        x = self.gap(x)                                            # (B, 64, 1, 1, 1)
        x = x.flatten(1)                                           # (B, 64)
        return self.fc(x)                                          # (B, 3)


engineering_probe = NonLocal3DCNN_MHA()
print(f"Engineering output shape: {engineering_probe(dummy_video).shape}")
print(f"Engineering params      : {count_parameters(engineering_probe):,}")

### 训练工程路径模型

现在训练高层 API 版本。

这一节不要只看最终精度，更要看它在教程中的角色：
- 学习路径告诉你“里面到底发生了什么”；
- 工程路径告诉你“理解之后怎样更高效地实现”。

由于我们已经刻意做了结构对齐：
- 通道压缩；
- 单头 attention；
- 零初始化输出投影；

因此这个版本既保留了工程写法的简洁性，
又不会和学习路径差得太远，适合作为“可比的工程近似实现”。

In [ ]:
print("=== Train engineering MHA model ===")
engineering_model = NonLocal3DCNN_MHA()
history_engineering = train_model(engineering_model, train_loader)
acc_engineering = evaluate_model(engineering_model, test_loader)
print(f"Engineering test accuracy: {acc_engineering:.2%}")

## v. 学习路径 vs 工程路径对比

到这里，建议你不要把两条路径理解成“二选一”，
而要把它们看成两个层次不同、但彼此补充的视角。

| 对比维度 | 学习路径 | 工程路径 |
|---|---|---|
| 教育价值 | 能直接看到 `theta/phi/g`、关系矩阵与残差怎样组成 Non-Local | 能快速掌握如何把全局关系写成标准 attention 模块 |
| 代码量 | 更长，显式控制每个步骤 | 更短，复用库实现 |
| 灵活性 | 可自由修改投影、归一化、相似度公式 | 受限于 `nn.MultiheadAttention` 接口 |
| 稳定性 | 更容易在 `transpose / reshape / softmax` 维度上出错 | 更稳定，经过广泛使用 |
| 工业适配度 | 更适合教学、论文复现、研究原型 | 更适合快速实验、工程集成 |
| 适用场景 | 面试准备、原理精读、调试 shape | 原型开发、模块替换、维护效率优先 |

### 一个更实用的结论

- 如果你现在还没有“真正看懂” Non-Local，先走学习路径；
- 如果你已经理解其核心思想，想更快做实验或接入项目，就走工程路径；
- 最佳顺序通常是：**先学透，再抽象；先手写，再复用。**

In [ ]:
plot_histories({
    "baseline": history_baseline,
    "learning_non_local": history_non_local,
    "engineering_mha": history_engineering,
})

fig, ax = plt.subplots(figsize=(7, 4))
model_names = ["Baseline", "Learning NL", "Engineering MHA"]
acc_values = [acc_baseline, acc_non_local, acc_engineering]
colors = ["#95a5a6", "#2ecc71", "#3498db"]
bars = ax.bar(model_names, acc_values, color=colors)
ax.set_title("Test Accuracy Comparison")
ax.set_ylabel("Accuracy")
ax.set_ylim(0, 1.05)
ax.grid(axis="y", alpha=0.3)
for bar, acc in zip(bars, acc_values):
    ax.text(bar.get_x() + bar.get_width() / 2, acc + 0.01, f"{acc:.2%}", ha="center")
plt.tight_layout()
plt.show()

## vi. 训练 vs 推理差异

Non-Local 与自回归模型不同，它没有“训练时 teacher forcing、推理时逐步生成”这种显式范式切换。
从算法角度看，训练和推理阶段走的是**同一套关系建模逻辑**。

真正的差异主要来自神经网络模块本身的运行状态：

| 阶段 | 学习路径行为 | 工程路径行为 |
|---|---|---|
| 训练 | `model.train()`，BatchNorm 使用当前 batch 统计量，参数参与反向传播 | 相同 |
| 推理 | `model.eval()`，BatchNorm 使用累计统计量，只保留前向计算 | 相同 |
| 全局关系计算 | 显式 `theta -> phi -> softmax -> g` | 封装在 `nn.MultiheadAttention` 内部 |

所以这里最值得记住的一句话是：
> **Non-Local 在训练和推理之间，变化的是模块状态，不是核心算法。**

In [ ]:
@torch.no_grad()
def inspect_predictions(model, loader, name, num_examples=6):
    model = model.to(device)
    model.eval()
    videos, labels = next(iter(loader))
    videos = videos.to(device)
    logits = model(videos)
    preds = logits.argmax(dim=1).cpu()

    print(name)
    for i in range(num_examples):
        print(f"  sample {i}: pred={preds[i].item()} | label={labels[i].item()}")

inspect_predictions(learning_non_local, test_loader, "Learning path predictions")
inspect_predictions(engineering_model, test_loader, "Engineering path predictions")

## vii. 结果与边界

教程类 notebook 的结果解读，不应只停留在“哪个数字更高”。
更重要的是：
**这些结果能支持我们理解什么，又不能支持我们断言什么。**

### 如何解读结果

- 如果 `learning_non_local` 明显优于 baseline，说明显式全局关系对当前跨帧模式识别是有帮助的；
- 如果 `engineering_mha` 与学习路径表现接近，说明高层 attention API 基本保留了 Non-Local 的核心收益；
- 如果差距不大，也并不反常，因为这里的数据规模很小、模式也相对规则。

### 学习路径的边界

- 优点：公式透明、shape 清晰、适合论文精读；
- 缺点：代码更长，更容易在 `transpose / view / softmax` 维度上犯错；
- 适合：面试、教学、论文复现、机制分析。

### 工程路径的边界

- 优点：代码更短、可维护性更好、更利于快速替换模块；
- 缺点：很多细节被封装，不适合逐式解释论文；
- 适合：工程实验、快速原型、现有模型改造。

### 最后要带走的认识

Non-Local 的价值不只是“多了一个模块”，
而是它把一个非常重要的建模思想说清楚了：
> **局部特征提取和全局关系建模，应该被拆成两个可以协作的子问题。**

## viii. 面试与项目表达

这一节的目标不是背题，而是帮你把“会写代码”转化成“能讲清楚”。

真正有区分度的面试表现，通常不是把术语堆出来，
而是能把下面三件事讲顺：
1. 这个模块为什么会被提出；
2. 它到底在计算什么；
3. 它在工程上值不值得用、该怎么用。

### 高频面试题

**Q1: Non-Local 和 Self-Attention 的关系是什么？**

- Embedded Gaussian 版本本质上就是视觉特征图上的自注意力；
- `theta/phi/g` 可以对应到 Query / Key / Value 的投影思想；
- 区别在于 Non-Local 从 `(T,H,W)` 特征图出发，而 Transformer 更常以 token 序列组织输入。

**Q2: 为什么 `W_z` 常初始化为 0？**

- 让模块初始输出近似等于输入；
- 插入已有网络时训练更稳定；
- 本质上是在保护残差主干，不让新分支一开始就破坏原特征。

**Q3: Non-Local 的主要瓶颈是什么？**

- `N x N` 关系矩阵带来二次复杂度；
- 当时间维或空间分辨率增大时，显存与算力开销都会迅速上升；
- 所以常见做法是在下采样后插入，或者对 `phi/g` 做池化近似。

**Q4: 为什么它比单纯堆卷积更适合长距离依赖？**

- 卷积依赖多层传播才能把远距离信息带过来；
- Non-Local 一层就能建立任意两位置的直接联系；
- 这对跨帧关联、远距离区域语义一致性特别重要。

**Q5: 工程里什么时候更适合用高层 attention API？**

- 当目标是快速验证模块思路；
- 当团队更关注维护效率而不是逐公式讲解；
- 当你希望复用底层优化实现时。

### 面试速答提纲

这一小节适合你在面试前快速扫一遍，用来把长答案压缩成一句可出口的表述。

| # | 问题 | 一句话回答 |
|---|---|---|
| 1 | Non-Local 在做什么？ | 让每个位置直接聚合所有位置特征，显式建模全局依赖。 |
| 2 | 为什么有效？ | 它绕过了卷积逐层传播的限制，一步建立长距离联系。 |
| 3 | 最大代价是什么？ | `N x N` 关系矩阵带来的二次复杂度。 |
| 4 | 为什么能即插即用？ | 输出 shape 不变，并且带残差连接。 |
| 5 | 为什么常放在下采样后？ | 为了降低全局关系计算的开销。 |
| 6 | 和 Transformer 的关系？ | 可视作视觉时空场景中的早期自注意力形式。 |

### 项目表达

> 如果面试官问“你做过什么相关项目”，可以这样组织回答：

- **学习深度**：我从零实现了 Non-Local Block，能解释 `theta/phi/g`、关系矩阵、残差连接与 shape 流转；
- **工程能力**：我又用 `nn.MultiheadAttention` 写了简洁版，实现相同的全局关系建模思想；
- **对比思考**：我能说清手写实现与高层实现的取舍，以及为什么要在下采样特征图上插入全局关系模块。

### 延伸阅读与对比

| 对比维度 | Non-Local | SE / CBAM | Transformer Self-Attention |
|---|---|---|---|
| 关注范围 | 全局时空位置 | 通道或局部空间重标定 | 全局 token 关系 |
| 计算粒度 | 像素 / 特征点级别 | 通道 / 空间权重 | token 级别 |
| 典型场景 | 视频、检测、分割 | CNN 增强模块 | 序列、视觉 Transformer |

如果把整份 notebook 压缩成一句话，它最适合表达的是：
> **我不仅知道 Non-Local 是什么，还知道它为什么重要、怎么实现、以及怎样把它迁移到工程写法里。**

## Appendix A. 可视化一个时空 query 的注意力范围

很多人第一次学 Non-Local，会在概念上知道“它能看全局”，
但脑子里并没有真正形成画面。

这个附录的目的就是把那个画面补上。

假设 query 位于 `(t=1, h=3, w=4)`：
- 普通卷积只会覆盖它附近的局部邻域；
- Non-Local 则允许它直接参考全部 `T * H * W` 位置。

这张图并不是严格的真实注意力权重可视化，
而是一个**教学示意图**：
用来帮助你直观区分“局部感受野”和“全局关系建模”之间的本质差异。

In [ ]:
T_vis, H_vis, W_vis = 4, 8, 8
q_t, q_h, q_w = 1, 3, 4

conv_scope = np.zeros((T_vis, H_vis, W_vis))
for dt in [-1, 0, 1]:
    for dh in [-1, 0, 1]:
        for dw in [-1, 0, 1]:
            tt = min(max(q_t + dt, 0), T_vis - 1)
            hh = min(max(q_h + dh, 0), H_vis - 1)
            ww = min(max(q_w + dw, 0), W_vis - 1)
            conv_scope[tt, hh, ww] = 1.0
conv_scope[q_t, q_h, q_w] = 2.0

non_local_scope = np.ones((T_vis, H_vis, W_vis))
non_local_scope[q_t, q_h, q_w] = 2.0

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(conv_scope.reshape(T_vis, -1).T, cmap="Blues", aspect="auto", vmin=0, vmax=2)
axes[0].set_title("Local Conv Receptive Scope")
axes[0].set_xlabel("Frame")
axes[0].set_ylabel("Flattened Spatial Index")

axes[1].imshow(non_local_scope.reshape(T_vis, -1).T, cmap="Purples", aspect="auto", vmin=0, vmax=2)
axes[1].set_title("Non-Local Attention Scope")
axes[1].set_xlabel("Frame")
axes[1].set_ylabel("Flattened Spatial Index")

plt.tight_layout()
plt.show()